RF, Anti-CCP → impute + add _was_missing flag (their missingness was class-dependent)
Everything else (ESR, CRP, HLA-B27, ANA, Anti-Ro, Anti-La, Anti-dsDNA, Anti-Sm, C3, C4) → straight MICE, no flag needed

In [1]:
import pandas as pd
import numpy as np

df = pd.read_excel("../data/raw/dataset.xlsx")
df.shape

(12085, 15)

In [2]:
# add missing flags because we are adding aditional column of was this originally missing needed for rf
df['RF_was_missing'] = df['RF'].isna().astype(int)
df['Anti-CCP_was_missing'] = df['Anti-CCP'].isna().astype(int)

df[['RF', 'RF_was_missing', 'Anti-CCP', 'Anti-CCP_was_missing']].head(10)

,RF,RF_was_missing,Anti-CCP,Anti-CCP_was_missing
0,34.2,0,29.9,0
1,35.5,0,28.9,0
2,21.3,0,21.3,0
3,26.0,0,39.0,0
4,38.1,0,30.8,0
5,NaN,1,37.3,0
6,22.3,0,NaN,1
7,31.8,0,38.1,0
8,33.4,0,NaN,1
9,37.4,0,25.0,0


In [3]:
df['RF_was_missing'].sum(), df['RF'].isna().sum()

(np.int64(1329), np.int64(1329))

In [4]:
#MICE needs  umeric so first convert positive/negative to 1/0
binary_cols = ['HLA-B27', 'ANA', 'Anti-Ro', 'Anti-La', 'Anti-dsDNA', 'Anti-Sm']

for col in binary_cols:
    df[col] = df[col].map({'Positive': 1, 'Negative': 0})

df[binary_cols].head(10)

,HLA-B27,ANA,Anti-Ro,Anti-La,Anti-dsDNA,Anti-Sm
0,1.0,0.0,1.0,0.0,1.0,1.0
1,0.0,NaN,1.0,NaN,1.0,NaN
2,0.0,0.0,NaN,1.0,0.0,NaN
3,NaN,NaN,1.0,1.0,NaN,NaN
4,1.0,0.0,1.0,0.0,1.0,0.0
5,1.0,NaN,0.0,0.0,NaN,0.0
6,0.0,1.0,0.0,NaN,1.0,1.0
7,0.0,1.0,0.0,0.0,1.0,1.0
8,NaN,NaN,1.0,NaN,1.0,NaN
9,1.0,0.0,0.0,NaN,1.0,1.0


In [5]:
df['Gender'] = df['Gender'].map({'Male': 1, 'Female': 0})

In [6]:
%pip install scikit-learn imbalanced-learn pyarrow

Note: you may need to restart the kernel to use updated packages.


In [7]:
from sklearn.model_selection import train_test_split

# split BEFORE imputation so no test-set information leaks into the MICE models
train_idx, test_idx = train_test_split(
    df.index, test_size=0.2, stratify=df['Disease'], random_state=42
)

df_train = df.loc[train_idx].copy()
df_test = df.loc[test_idx].copy()

print(df_train.shape, df_test.shape)

(9668, 17) (2417, 17)


In [8]:
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer

mice_cols = ['ESR', 'CRP', 'RF', 'Anti-CCP', 'HLA-B27', 'ANA',
             'Anti-Ro', 'Anti-La', 'Anti-dsDNA', 'Anti-Sm', 'C3', 'C4']

imputer = IterativeImputer(random_state=42, max_iter=50)  # 15 was too low: doesn't converge (verified converges at iter 17)
df_train[mice_cols] = imputer.fit_transform(df_train[mice_cols])
df_test[mice_cols] = imputer.transform(df_test[mice_cols])  # transform only, never fit on test

df_train[mice_cols].isna().sum(), df_test[mice_cols].isna().sum()

(ESR           0
 CRP           0
 RF            0
 Anti-CCP      0
 HLA-B27       0
 ANA           0
 Anti-Ro       0
 Anti-La       0
 Anti-dsDNA    0
 Anti-Sm       0
 C3            0
 C4            0
 dtype: int64,
 ESR           0
 CRP           0
 RF            0
 Anti-CCP      0
 HLA-B27       0
 ANA           0
 Anti-Ro       0
 Anti-La       0
 Anti-dsDNA    0
 Anti-Sm       0
 C3            0
 C4            0
 dtype: int64)

In [9]:
# NOTE: previously this rounded MICE's continuous probability estimate to a hard 0/1.
# Investigation (see project notes) showed this destroys real information: for markers
# with a true ~50% base rate in non-associated diseases (e.g. ANA in PsA), rounding
# manufactured a fabricated strong association (46.8% true rate -> 18.6% after rounding)
# that carries no real clinical signal. Empirically, keeping the continuous [0,1] MICE
# estimate improved Random Forest Macro-F1 by +0.0064 and specifically improved recall
# on AS, Normal, and Sjogren's -- so we clip to a valid probability range but do NOT
# round to a hard binary value.
for col in binary_cols:
    df_train[col] = df_train[col].clip(0, 1)
    df_test[col] = df_test[col].clip(0, 1)

df_train[binary_cols].describe()

,HLA-B27,ANA,Anti-Ro,Anti-La,Anti-dsDNA,Anti-Sm
count,9668.000000,9668.000000,9668.000000,9668.000000,9668.000000,9668.000000
mean,0.613619,0.628544,0.575924,0.588238,0.551940,0.554096
std,0.447271,0.413501,0.435301,0.431291,0.397798,0.384357
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,1.000000,0.807538,0.661116,0.683380,0.558158,0.558674
75%,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000
max,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000


In [10]:
continuous_cols = ['ESR', 'CRP', 'RF', 'Anti-CCP', 'C3', 'C4']
df_train[continuous_cols].describe()

,ESR,CRP,RF,Anti-CCP,C3,C4
count,9668.000000,9668.000000,9668.000000,9668.000000,9668.000000,9668.000000
mean,24.221309,13.313933,19.722205,19.677782,131.658827,38.163100
std,14.188548,10.077268,10.960164,10.278980,34.346890,18.602235
min,0.000000,-3.777017,0.000000,0.000000,50.000000,5.000000
25%,10.000000,2.100000,10.634469,12.105162,107.000000,24.000000
50%,28.000000,15.700000,19.300000,19.400000,133.000000,38.109556
75%,36.000000,21.800000,28.687382,27.116405,157.000000,52.454562
max,49.000000,30.699155,40.000000,40.000000,205.000000,74.000000


In [11]:
for col in continuous_cols:
    df_train[col] = df_train[col].clip(lower=0)
    df_test[col] = df_test[col].clip(lower=0)

df_train[continuous_cols].describe()

,ESR,CRP,RF,Anti-CCP,C3,C4
count,9668.000000,9668.000000,9668.000000,9668.000000,9668.000000,9668.000000
mean,24.221309,13.334035,19.722205,19.677782,131.658827,38.163100
std,14.188548,10.048652,10.960164,10.278980,34.346890,18.602235
min,0.000000,0.000000,0.000000,0.000000,50.000000,5.000000
25%,10.000000,2.100000,10.634469,12.105162,107.000000,24.000000
50%,28.000000,15.700000,19.300000,19.400000,133.000000,38.109556
75%,36.000000,21.800000,28.687382,27.116405,157.000000,52.454562
max,49.000000,30.699155,40.000000,40.000000,205.000000,74.000000


In [12]:
df_train[binary_cols].apply(pd.Series.value_counts)

,HLA-B27,ANA,Anti-Ro,Anti-La,Anti-dsDNA,Anti-Sm
0.000000,3131.0,2469.0,3122.0,2998.0,2649.0,2455.0
0.173851,NaN,1.0,NaN,NaN,NaN,NaN
0.185793,NaN,1.0,NaN,NaN,NaN,NaN
0.199746,NaN,1.0,NaN,NaN,NaN,NaN
0.205776,NaN,NaN,NaN,NaN,1.0,NaN
...,...,...,...,...,...,...
0.997860,NaN,1.0,NaN,NaN,NaN,NaN
0.999096,NaN,1.0,NaN,NaN,NaN,NaN
0.999183,NaN,1.0,NaN,NaN,NaN,NaN
0.999649,NaN,1.0,NaN,NaN,NaN,NaN


In [13]:
for d in (df_train, df_test):
    # Inflammation_Score = ESR + CRP: an ad-hoc composite of two inflammatory markers,
    # NOT a validated clinical scoring system. In the paper, refer to this as an
    # "ESR-CRP composite feature," not a clinical "score," unless a cited source
    # supports that terminology.
    d['Inflammation_Score'] = d['ESR'] + d['CRP']

    # C3_C4_Ratio: exploratory engineered feature — complement C3:C4 ratio.
    # Shows a notably different mean for SLE vs other classes in EDA; introduced as a
    # hypothesis-driven feature (complement consumption patterns differ in SLE), not
    # purely because it happened to improve validation score.
    # Defensive: .replace(0, pd.NA) prevents a silent division-by-zero -> inf if C4 is
    # ever imputed as slightly negative and clipped to exactly 0 (hasn't happened with
    # current data — verified min C4 = 5.0 in raw data — but MICE could theoretically
    # predict this for future/different data).
    d['C3_C4_Ratio'] = d['C3'] / d['C4'].replace(0, pd.NA)

    # Autoantibody_Count: count of positive results among the 6 binary autoantibody
    # markers. Exploratory aggregate feature, not a validated diagnostic index.
    d['Autoantibody_Count'] = d[binary_cols].sum(axis=1)

df_train[['Inflammation_Score', 'C3_C4_Ratio', 'Autoantibody_Count']].describe()

,Inflammation_Score,C3_C4_Ratio,Autoantibody_Count
count,9668.000000,9668.000000,9668.000000
mean,37.555344,4.693479,3.512362
std,23.588803,3.140674,1.102248
min,0.000000,1.229730,0.000000
25%,12.200000,2.658530,2.735463
50%,45.600000,3.548387,3.548895
75%,57.500000,5.556324,4.336764
max,79.699155,21.415833,6.000000


In [14]:
df_train.groupby('Disease')[['Inflammation_Score', 'C3_C4_Ratio', 'Autoantibody_Count']].mean().round(2)

,Inflammation_Score,C3_C4_Ratio,Autoantibody_Count
Disease,,,
Ankylosing Spondylitis,53.39,3.96,3.44
Normal,10.92,4.09,3.25
Psoriatic Arthritis,62.11,3.98,2.97
Reactive Arthritis,40.48,4.02,3.53
Rheumatoid Arthritis,55.40,4.07,3.07
Sjögren's Syndrome,10.51,4.01,4.38
Systemic Lupus Erythematosus,10.26,10.00,4.38


In [15]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
df_train['Disease_encoded'] = le.fit_transform(df_train['Disease'])
df_test['Disease_encoded'] = le.transform(df_test['Disease'])

# check the mapping so we can decode predictions back to disease names later
dict(zip(le.classes_, le.transform(le.classes_)))

{'Ankylosing Spondylitis': np.int64(0),
 'Normal': np.int64(1),
 'Psoriatic Arthritis': np.int64(2),
 'Reactive Arthritis': np.int64(3),
 'Rheumatoid Arthritis': np.int64(4),
 "Sjögren's Syndrome": np.int64(5),
 'Systemic Lupus Erythematosus': np.int64(6)}

In [16]:
# Stage 1 feature set: original 14 raw features only
raw_features = ['Age', 'Gender', 'ESR', 'CRP', 'RF', 'Anti-CCP', 'HLA-B27',
                 'ANA', 'Anti-Ro', 'Anti-La', 'Anti-dsDNA', 'Anti-Sm', 'C3', 'C4']

# Stage 1b feature set: raw + missingness flags + engineered features
extended_features = raw_features + ['RF_was_missing', 'Anti-CCP_was_missing',
                                     'Inflammation_Score', 'C3_C4_Ratio', 'Autoantibody_Count']

X_raw_train, X_raw_test = df_train[raw_features], df_test[raw_features]
X_ext_train, X_ext_test = df_train[extended_features], df_test[extended_features]
y_train, y_test = df_train['Disease_encoded'], df_test['Disease_encoded']

print(X_raw_train.shape, X_raw_test.shape)
print(X_ext_train.shape, X_ext_test.shape)

(9668, 14) (2417, 14)
(9668, 19) (2417, 19)


In production, your model receives a single new user, a new sensor ping, or a new transaction. You cannot calculate the mean or standard deviation of data you haven't received yet. Therefore, any step that uses the test set to determine how to process the training set cheats this simulation.

In [17]:
from sklearn.preprocessing import StandardScaler

continuous_cols = ['Age', 'ESR', 'CRP', 'RF', 'Anti-CCP', 'C3', 'C4']
extended_continuous_cols = continuous_cols + ['Inflammation_Score', 'C3_C4_Ratio']

extended_continuous_cols = continuous_cols + ['Inflammation_Score', 'C3_C4_Ratio', 'Autoantibody_Count']

scaler_raw = StandardScaler()
X_raw_train_scaled = X_raw_train.copy()
X_raw_test_scaled = X_raw_test.copy()
X_raw_train_scaled[continuous_cols] = scaler_raw.fit_transform(X_raw_train[continuous_cols])
X_raw_test_scaled[continuous_cols] = scaler_raw.transform(X_raw_test[continuous_cols])

scaler_ext = StandardScaler()
X_ext_train_scaled = X_ext_train.copy()
X_ext_test_scaled = X_ext_test.copy()
X_ext_train_scaled[extended_continuous_cols] = scaler_ext.fit_transform(X_ext_train[extended_continuous_cols])
X_ext_test_scaled[extended_continuous_cols] = scaler_ext.transform(X_ext_test[extended_continuous_cols])

X_raw_train_scaled.head()

,Age,Gender,ESR,CRP,RF,Anti-CCP,HLA-B27,ANA,Anti-Ro,Anti-La,Anti-dsDNA,Anti-Sm,C3,C4
327,0.908935,0,-0.017032,-0.092956,0.727928,1.121007,0.0,1.000000,0.000000,0.616933,0.567717,0.000000,0.068166,0.069828
10698,1.473599,0,1.675994,1.249383,-1.562302,-1.048581,0.0,1.000000,0.484713,0.462812,1.000000,0.000000,0.679607,0.098751
6344,0.061939,1,-0.579463,-1.118022,1.275393,-0.114588,0.0,0.859225,1.000000,0.757306,0.610198,1.000000,-0.219666,0.206271
4506,-1.462653,0,0.477783,1.200817,-0.786727,-0.211879,1.0,0.524279,1.000000,0.479366,1.000000,0.481931,-0.601507,1.173947
5436,1.021867,0,0.759715,0.245415,0.819172,0.615095,1.0,0.626255,0.000000,1.000000,1.000000,1.000000,-0.048299,0.528829


Cleaned, imputed data (MICE for most features, missing-indicator flags for RF/Anti-CCP)
Engineered features computed and held aside (Inflammation_Score, C3_C4_Ratio, Autoantibody_Count)
Two scaled feature sets with matching train/test rows: X_raw_train_scaled/X_raw_test_scaled (14 features, for Stage 1) and X_ext_train_scaled/X_ext_test_scaled (19 features, for the engineered-features comparison)
Encoded label y_train/y_test with the class mapping saved

In [18]:
import os

os.makedirs('../data/processed', exist_ok=True)

# raw feature set (Stage 1)
X_raw_train_scaled.to_csv('../data/processed/X_raw_train.csv', index=True)
X_raw_test_scaled.to_csv('../data/processed/X_raw_test.csv', index=True)

# extended feature set (Stage 1b comparison)
X_ext_train_scaled.to_csv('../data/processed/X_ext_train.csv', index=True)
X_ext_test_scaled.to_csv('../data/processed/X_ext_test.csv', index=True)

# labels (shared across both feature sets, same rows)
y_train.to_csv('../data/processed/y_train.csv', index=True)
y_test.to_csv('../data/processed/y_test.csv', index=True)

print("Saved all processed files to data/processed/")

Saved all processed files to data/processed/
